In [1]:
import time

# We will need Qibo to write our quantum circuit
from qibo import Circuit, gates, set_backend

# We will need mpstab to construct a surrogate
from mpstab import HSMPO

In [2]:
n = 10

circ = Circuit(nqubits=n)
circ.add(gates.H(0))
[circ.add(gates.CNOT(q, q+1)) for q in range(n - 1)]

circ.draw()

0: ─H─o─────────────────
1: ───X─o───────────────
2: ─────X─o─────────────
3: ───────X─o───────────
4: ─────────X─o─────────
5: ───────────X─o───────
6: ─────────────X─o─────
7: ───────────────X─o───
8: ─────────────────X─o─
9: ───────────────────X─


In [ ]:
surry = HSMPO(circ)

In [ ]:
# %time
surry.expectation(
    observable="Z"*n
)

In [2]:
from mpstab.models.ansatze import HardwareEfficient 
from mpstab.utils import obs_string_to_qibo_hamiltonian

def execute_circuit(nqubits, nlayers):

    set_backend("numpy")
    obs_str = "Z" * nqubits

    ans = HardwareEfficient(nqubits=nqubits, nlayers=nlayers)
    hs = HSMPO(ansatz=ans)

    it = time.time()
    expval_mpstab = hs.expectation(obs_str)
    time_mpstab = time.time() - it
    
    ham = obs_string_to_qibo_hamiltonian(obs_str)
    it = time.time()
    expval_qibo = ham.expectation_from_state(
        ans.circuit().state()
    )
    time_qibo = time.time() - it
    return (expval_mpstab, expval_qibo), (time_mpstab, time_qibo)

In [8]:
(expval_mpstab, expval_qibo), (time_mpstab, time_qibo) = execute_circuit(nqubits=12, nlayers=2)

print(f"MPSTAB expectation value: {expval_mpstab}, time taken: {time_mpstab:.4f} seconds")
print(f"Qibo expectation value: {expval_qibo}, time taken: {time_qibo:.4f} seconds")

[Qibo 0.3.0|INFO|2026-02-15 23:48:43]: Using numpy backend on /CPU:0


MPSTAB expectation value: 0.0006079123653988358, time taken: 0.5040 seconds
Qibo expectation value: 0.0006079270405313344, time taken: 20.8455 seconds
